# **Validasi Pasangan EN–ID | MASSIVE-Agents**

---

**Project:** KP Agentic AI + LLM  
**Tahap:** Validasi pasangan bahasa (Step 8)  
**Dataset:** `development` · `en-US` dan `id-ID`

Pasangan dibentuk menggunakan `source_record_id` yang sama. Pemeriksaan otomatis digunakan untuk melihat konsistensi ID, grup, fungsi target, dan struktur GT. Kesetaraan makna instruksi tetap memerlukan penilaian manual.

## **1 · Dataset dan Pengaturan**

---

Data bersumber dari hasil preprocessing yang telah direview. Pemeriksaan dibatasi pada split `development` agar data `test` tidak digunakan untuk pengembangan metode.

In [1]:
from pathlib import Path
from collections import Counter
from datetime import datetime, timezone
import csv
import hashlib
import json
import os
import random

RANDOM_SEED = 20261110
JUMLAH_SAMPEL = 100

# Folder utama project yang berisi data/processed/massive_agents_reviewed
PROJECT_ROOT_OVERRIDE = None

def locate_project():
    options = [Path(PROJECT_ROOT_OVERRIDE)] if PROJECT_ROOT_OVERRIDE else []
    options += [Path.cwd(), *Path.cwd().parents, Path(r"D:\KP-Agentic-AI")]
    for folder in options:
        if (folder / "data" / "processed" / "massive_agents_reviewed" / "en-US_development.jsonl").is_file():
            return folder
    raise FileNotFoundError("Dataset development tidak ditemukan pada folder project.")

ROOT = locate_project()
DATA_DIR = ROOT / "data" / "processed" / "massive_agents_reviewed"
OUTPUT_DIR = ROOT / "results" / "validasi_pasangan_en_id"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

PATH_EN = DATA_DIR / "en-US_development.jsonl"
PATH_ID = DATA_DIR / "id-ID_development.jsonl"
print("Project:", ROOT)
print("Dataset:", DATA_DIR)
print("Output:", OUTPUT_DIR)

Project: d:\KP-Agentic-AI
Dataset: d:\KP-Agentic-AI\data\processed\massive_agents_reviewed
Output: d:\KP-Agentic-AI\results\validasi_pasangan_en_id


## **2 · Pembacaan dan Pencocokan ID**

---

Setiap `source_record_id` dicocokkan antara bahasa Inggris dan bahasa Indonesia. Pasangan harus berasal dari `group_id` dan split yang sama. Fungsi target juga diperiksa tanpa mengubah GT.

In [2]:
def read_jsonl(path):
    records = []
    with path.open(encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"JSON tidak valid pada {path.name}:{line_no}") from e
    return records

def index_by_id(records, language):
    by_id = {}
    for rec in records:
        key = str(rec["source_record_id"])
        if key in by_id:
            raise ValueError(f"Duplikasi source_record_id {key} pada {language}")
        if rec["split"] != "development" or rec["language_code"] != language:
            raise ValueError(f"Split atau bahasa tidak sesuai pada {language}:{key}")
        by_id[key] = rec
    return by_id

en_records = read_jsonl(PATH_EN)
id_records = read_jsonl(PATH_ID)
en_by_id = index_by_id(en_records, "en-US")
id_by_id = index_by_id(id_records, "id-ID")

paired_ids = sorted(set(en_by_id) & set(id_by_id), key=lambda x: (not x.isdigit(), int(x) if x.isdigit() else x))
unpaired_en = set(en_by_id) - set(id_by_id)
unpaired_id = set(id_by_id) - set(en_by_id)

assert len(paired_ids) == 397, f"Jumlah pasangan berbeda dari acuan: {len(paired_ids)}"
assert all(en_by_id[k]["group_id"] == id_by_id[k]["group_id"] for k in paired_ids)
assert all(len(en_by_id[k]["gt"]) == len(id_by_id[k]["gt"]) == 1 for k in paired_ids)
assert all(next(iter(en_by_id[k]["gt"])) == next(iter(id_by_id[k]["gt"])) for k in paired_ids)

print("EN development:", len(en_records))
print("ID development:", len(id_records))
print("Pasangan ID:", len(paired_ids))
print("EN tanpa pasangan:", len(unpaired_en))
print("ID tanpa pasangan:", len(unpaired_id))
print("Group ID dan fungsi target: sesuai")

EN development: 582
ID development: 397
Pasangan ID: 397
EN tanpa pasangan: 185
ID tanpa pasangan: 0
Group ID dan fungsi target: sesuai


## **3 · Pemeriksaan Ground Truth Pasangan**

---

GT identik secara teks tidak menjamin kedua instruksi memiliki makna yang sama. Perbedaan nilai GT juga belum tentu merupakan kesalahan karena nama tempat, waktu, atau entitas dapat diterjemahkan.

Hasil otomatis dibagi menjadi `GT identik`, `nilai GT berbeda`, dan `parameter GT berbeda`. Ketiganya masih memerlukan validasi semantik.

In [3]:
rows = []
for source_id in paired_ids:
    en = en_by_id[source_id]
    ind = id_by_id[source_id]
    fn = next(iter(en["gt"]))
    args_en = en["gt"][fn]
    args_id = ind["gt"][fn]
    keys_en = set(args_en)
    keys_id = set(args_id)

    if en["gt"] == ind["gt"]:
        category = "GT identik"
    elif keys_en != keys_id:
        category = "parameter GT berbeda"
    else:
        category = "nilai GT berbeda"

    rows.append({
        "source_record_id": source_id,
        "group_id": en["group_id"],
        "split": "development",
        "row_key_en": en["row_key"],
        "row_key_id": ind["row_key"],
        "question_en": en["question"],
        "question_id": ind["question"],
        "target_function": fn,
        "gt_en": json.dumps(en["gt"], ensure_ascii=False, sort_keys=True),
        "gt_id": json.dumps(ind["gt"], ensure_ascii=False, sort_keys=True),
        "kategori_gt_otomatis": category,
        "parameter_hanya_en": ", ".join(sorted(keys_en - keys_id)),
        "parameter_hanya_id": ", ".join(sorted(keys_id - keys_en)),
        "setara_semantik": "BELUM_DIREVIEW",
        "entitas_dan_nilai_setara": "BELUM_DIREVIEW",
        "gt_sesuai_instruksi": "BELUM_DIREVIEW",
        "catatan_review": "",
    })

counts = Counter(r["kategori_gt_otomatis"] for r in rows)
print("Kategori otomatis (belum keputusan semantik):")
for key in ("GT identik", "nilai GT berbeda", "parameter GT berbeda"):
    print(f"- {key}: {counts[key]}")

Kategori otomatis (belum keputusan semantik):
- GT identik: 109
- nilai GT berbeda: 288
- parameter GT berbeda: 0


## **4 · Sampel Validasi Manual**

---

Sebanyak 100 pasangan dipilih secara purposive dan terstratifikasi. Sampel lebih banyak mengambil pasangan dengan GT berbeda agar potensi masalah terjemahan dan anotasi dapat diperiksa. Hasil sampel tidak digunakan untuk memperkirakan prevalensi seluruh dataset.

Kolom manual menggunakan `YA`, `TIDAK`, atau `RAGU`. Nilai awal tetap `BELUM_DIREVIEW`.

In [4]:
rng = random.Random(RANDOM_SEED)
by_category = {name: [r for r in rows if r["kategori_gt_otomatis"] == name] for name in counts}
quotas = {"GT identik": 25, "nilai GT berbeda": 65, "parameter GT berbeda": 10}

sample = []
for category, quota in quotas.items():
    pool = by_category.get(category, [])
    selected = rng.sample(pool, min(len(pool), quota))
    sample.extend(selected)

if len(sample) < JUMLAH_SAMPEL:
    remaining = [r for r in rows if r["source_record_id"] not in {s["source_record_id"] for s in sample}]
    sample.extend(rng.sample(remaining, JUMLAH_SAMPEL - len(sample)))

sample.sort(key=lambda r: (r["kategori_gt_otomatis"], int(r["source_record_id"])))
assert len(sample) == JUMLAH_SAMPEL
assert len({s["source_record_id"] for s in sample}) == JUMLAH_SAMPEL
print("Sampel validasi:", len(sample))
print("Sebaran:", dict(Counter(r["kategori_gt_otomatis"] for r in sample)))

Sampel validasi: 100
Sebaran: {'GT identik': 28, 'nilai GT berbeda': 72}


## **5 · Penyimpanan Hasil**

---

Seluruh pasangan dan sampel audit disimpan terpisah. Data, label GT, dan keputusan preprocessing asli tidak diubah. Metadata mencatat jumlah data, konfigurasi sampling, serta checksum file sumber.

In [5]:
def write_csv(path, content):
    manual_cols = ("setara_semantik", "entitas_dan_nilai_setara", "gt_sesuai_instruksi", "catatan_review")
    if path.is_file():
        # Hasil penilaian manual sebelumnya tidak ditimpa saat notebook dijalankan ulang.
        with path.open(encoding="utf-8-sig", newline="") as f:
            previous = {r["source_record_id"]: r for r in csv.DictReader(f)}
        for item in content:
            prev = previous.get(item["source_record_id"], {})
            for col in manual_cols:
                if prev.get(col) and prev[col] != "BELUM_DIREVIEW":
                    item[col] = prev[col]
    with path.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(content[0].keys()))
        writer.writeheader()
        writer.writerows(content)

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(2 ** 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

all_path = OUTPUT_DIR / "pasangan_en_id_development_397.csv"
sample_path = OUTPUT_DIR / "sampel_validasi_en_id_100.csv"
meta_path = OUTPUT_DIR / "metadata_validasi_en_id.json"

write_csv(all_path, rows)
write_csv(sample_path, sample)
metadata = {
    "run": "validasi_pasangan_en_id_step8_v1",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "split": "development",
    "source_en_sha256": sha256(PATH_EN),
    "source_id_sha256": sha256(PATH_ID),
    "en_rows": len(en_records),
    "id_rows": len(id_records),
    "paired_rows": len(rows),
    "unpaired_en": len(unpaired_en),
    "unpaired_id": len(unpaired_id),
    "gt_categories": dict(counts),
    "sample_size": len(sample),
    "sample_seed": RANDOM_SEED,
    "sample_method": "purposive-stratified; oversample GT differences; not prevalence estimate",
    "semantic_review": "pending; automatic GT equality is not semantic equivalence",
    "gt_modified": False,
    "test_set_used": False,
}
meta_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print("Output:")
for f in (all_path, sample_path, meta_path):
    print("-", f)

Output:
- d:\KP-Agentic-AI\results\validasi_pasangan_en_id\pasangan_en_id_development_397.csv
- d:\KP-Agentic-AI\results\validasi_pasangan_en_id\sampel_validasi_en_id_100.csv
- d:\KP-Agentic-AI\results\validasi_pasangan_en_id\metadata_validasi_en_id.json


## **6 · Ringkasan**

---

Pasangan EN–ID sudah cocok berdasarkan ID, grup, dan fungsi target. Validasi makna tetap dilakukan pada teks instruksi serta nilai argumen. Analisis perbedaan performa bahasa belum dapat ditafsirkan sebagai dampak bahasa saja sebelum penilaian semantik selesai.

In [6]:
print(f"Pasangan development: {len(rows)}")
print(f"GT identik: {counts['GT identik']}")
print(f"Nilai GT berbeda: {counts['nilai GT berbeda']}")
print(f"Parameter GT berbeda: {counts['parameter GT berbeda']}")
print(f"Sampel manual: {len(sample)} (belum direview)")
print("Test set tidak digunakan")
print("Contoh pasangan:")
for r in sample[:3]:
    print(f"[{r['source_record_id']}] EN: {r['question_en']}")
    print(f"       ID: {r['question_id']}")

Pasangan development: 397
GT identik: 109
Nilai GT berbeda: 288
Parameter GT berbeda: 0
Sampel manual: 100 (belum direview)
Test set tidak digunakan
Contoh pasangan:
[1360] EN: please raise the light i am not comfortable at this
       ID: tolong cerahakan lampu karena saya tidak nyaman dengan ini
[1993] EN: can you turn up the lights
       ID: bisakah kamu nyalakan lampunya
[2093] EN: bring me the title of current music
       ID: beritahu saya judul musik saat ini
